# AttackAware PolyIoM v1.1.4 — post-revocation acceptance

The inversion run showed every arm is fully invertible (SAR 100%), that the
polynomial does protect the raw embedding (cosine 0.222 against 0.913 for
IoM-GRP alone on voice), but that the reconstruction still matched the true
subject at **3.0× the threshold** after the IoM projection was re-keyed.

That run re-keyed `R` only. This one re-keys **everything each arm has**:

| arm | re-issued on revocation |
|---|---|
| `polyiom` | fresh polynomial key `K**` **and** fresh projection `R'` |
| `randproj_iom` | fresh random map `A'` **and** fresh projection `R'` |
| `iom_only` | fresh projection `R'` (it has nothing else) |

**The attacker never re-attacks.** They hold one stolen template, reconstruct
once, and the subject then re-enrols under the new keys. That is the real
revocation scenario.

**Headline metric: PRAR** — post-revocation acceptance rate, the fraction of
subjects whose *old* template's reconstruction is still accepted at the sealed
threshold after full re-keying. **Revocation works only if PRAR falls to near
zero.** A PRAR near 100% means one stolen template is a permanent credential.

This run also bootstraps the cosine contrast the inversion run could only
report as a point estimate, so it becomes firm or not.

**Construction of `K**`:** resampled from `K*`'s own coefficient and exponent
values, *not* re-run through Stage A's inversion gate. For this question that
is the right construction — we are asking whether `z'` resembles `z` in the
ways any polynomial of that family sees, not whether `K**` is a good key.
Three independent draws guard against a fluke, and a degeneracy check rejects
any draw whose re-keyed system stops discriminating between subjects.

Reads the stored embeddings, keys and sealed results; writes one new file,
`runs/revocation/revocation_result.json`, and no seal.


In [ ]:
#@title 1. Mount Drive and verify the revocation inputs
from google.colab import drive
drive.mount("/content/drive")

import hashlib, json, math, os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/AttackAware_PolyIoM_v1_1_4")
DIR = {
    "protocol": PROJECT / "protocol",
    "embeddings": PROJECT / "embeddings",
    "runs": PROJECT / "runs",
    "seal": PROJECT / "seal",
}
S0, G = 2026, 5
DEVICE = torch.device("cpu")

LFW_FACE_TRIALS = DIR["protocol"] / "lfw_face_trials.tsv"
LFW_EMB = DIR["embeddings"] / "lfw_all_valid_embeddings.npz"
LIBRI_MANIFEST = DIR["protocol"] / "librispeech_internal.tsv"
LIBRI_EMB = DIR["embeddings"] / "librispeech_internal_embeddings.npz"

HELDOUT_RESULTS = {
    modality: DIR["runs"] / "heldout" / modality / "heldout_result.json"
    for modality in ("face", "voice")
}

REQUIRED = [
    LFW_FACE_TRIALS, LFW_EMB, LIBRI_MANIFEST, LIBRI_EMB,
    DIR["runs"] / "key_search/face/selected_key.json",
    DIR["runs"] / "key_search/voice/selected_key.json",
    HELDOUT_RESULTS["face"], HELDOUT_RESULTS["voice"],
    PROJECT / "ablation_only.py",
    PROJECT / "inversion_only.py",
    PROJECT / "revocation_only.py",
]
missing = [str(p.relative_to(PROJECT)) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Revocation cannot start; required artefacts are missing:\n- "
        + "\n- ".join(missing)
    )

torch.use_deterministic_algorithms(True, warn_only=False)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print("Preflight: PASS")
print("Writes exactly one file: runs/revocation/revocation_result.json")
print("Changes no seal and fits no threshold.")

In [ ]:
#@title 2. Load the pipeline, the attack and the revocation test
for name in ("ablation_only.py", "inversion_only.py", "revocation_only.py"):
    path = PROJECT / name
    exec(compile(path.read_text(), str(path), "exec"), globals())
print("Revocation runtime: READY")

In [ ]:
#@title 3. Re-run the attack, then re-key everything and re-test
revocation = run_revocation()
revocation_report(revocation)